In [1]:
import obspy
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import swspy

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


In [ ]:
# Import the earthquake catalog
#catalog = pd.read_csv('../data/processed_catalog.csv')

print("Earthquake Catalog:")
print(f"Shape: {catalog.shape}")
print("\nFirst few rows:")
print(catalog.head())
print(f"\nDate range: {catalog['year'].min()}-{catalog['year'].max()}")
print(f"Number of events: {len(catalog)}")

Earthquake Catalog:
Shape: (144328, 18)

First few rows:
   year  month  day  hour  minute  second       lat        lon    dep    eh1  \
0  2014     11    4    20      46   1.620  45.93439 -130.02092  1.148  0.432   
1  2014     11    4    22      12  40.197  45.92662 -130.01597  0.904  0.089   
2  2014     11    4    22      41   7.813  45.92568 -129.98059  0.821  0.016   
3  2014     11    5     0      43   9.964  45.94480 -129.99482  1.140  0.222   
4  2014     11    5     1       3  48.101  45.93533 -129.98498  0.559 -1.000   

     eh2   az     ez  mag       id  month_num  day_num        date  
0  0.023  168  0.030  1.5  1000002         11        4  2014-11-01  
1  0.027  114  0.039  0.2  1000010         11        4  2014-11-01  
2  0.000  146  0.003  0.4  1000012         11        4  2014-11-01  
3  0.030   87  0.698 -0.0  1000025         11        5  2014-11-01  
4 -1.000   -1 -1.000 -0.1  1000030         11        5  2014-11-01  

Date range: 2014-2021
Number of events: 144328


In [3]:
# Now let's create the correct parsing function based on the observed format
def parse_phase_file_correct(filename):
    """
    Parse the phase file with the correct format:
    Event lines: # YYYY MM DD HH MM SS.ss LAT LON DEPTH MAG ... EVENT_ID E
    Phase lines: STATION ARRIVAL_TIME WEIGHT PHASE_TYPE QUALITY
    
    Returns:
    --------
    events_df : pandas.DataFrame
        DataFrame with earthquake event information
    phases_df : pandas.DataFrame  
        DataFrame with phase picks for each event
    """
    
    events = []
    phases = []
    current_event_id = None
    current_event_info = None
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    print(f"Processing {len(lines)} lines...")
    
    for line_num, line in enumerate(lines):
        stripped = line.strip()
        
        # Skip empty lines
        if not stripped:
            continue
            
        parts = stripped.split()
        
        # Check if this is an event line (starts with #)
        if stripped.startswith('#') and len(parts) >= 10:
            # Event line format: # YYYY MM DD HH MM SS.ss LAT LON DEPTH MAG ... EVENT_ID E
            try:
                year = int(parts[1])
                month = int(parts[2])
                day = int(parts[3])
                hour = int(parts[4])
                minute = int(parts[5])
                second = float(parts[6])
                lat = float(parts[7])
                lon = float(parts[8])
                depth = float(parts[9])
                
                # Find the event ID (second to last element, before 'E')
                event_id = parts[-2] if len(parts) >= 2 else str(len(events))
                
                # Create UTC datetime string
                datetime_str = f"{year:04d}-{month:02d}-{day:02d}T{hour:02d}:{minute:02d}:{second:06.3f}Z"
                
                event_info = {
                    'event_id': event_id,
                    'year': year,
                    'month': month,
                    'day': day, 
                    'hour': hour,
                    'minute': minute,
                    'second': second,
                    'datetime_str': datetime_str,
                    'lat': lat,
                    'lon': lon,
                    'depth': depth,
                    'line_number': line_num + 1,
                    'raw_line': stripped
                }
                
                # Add magnitude if available (typically at index 10)
                if len(parts) > 10:
                    try:
                        event_info['magnitude'] = float(parts[10])
                    except ValueError:
                        event_info['magnitude'] = np.nan
                
                events.append(event_info)
                current_event_info = event_info
                current_event_id = event_id
                
            except (ValueError, IndexError) as e:
                print(f"Warning: Could not parse event line {line_num + 1}: {stripped}")
                continue
                
        else:
            # This should be a phase pick line if we have a current event
            if current_event_id is not None and current_event_info is not None and len(parts) >= 4:
                # Phase line format: STATION ARRIVAL_TIME WEIGHT PHASE_TYPE QUALITY
                try:
                    phase_info = {
                        'event_id': current_event_id,
                        'event_datetime': current_event_info['datetime_str'],
                        'event_lat': current_event_info['lat'],
                        'event_lon': current_event_info['lon'],
                        'event_depth': current_event_info['depth'],
                        'station': parts[0],
                        'arrival_time': float(parts[1]),
                        'weight': float(parts[2]) if parts[2] != '-1.000' else np.nan,
                        'phase_type': parts[3],
                        'quality': parts[4] if len(parts) > 4 else '',
                        'line_number': line_num + 1,
                        'raw_line': stripped
                    }
                    
                    phases.append(phase_info)
                    
                except (ValueError, IndexError) as e:
                    # Skip problematic phase lines
                    continue
    
    # Convert to DataFrames
    events_df = pd.DataFrame(events)
    phases_df = pd.DataFrame(phases)
    
    # Convert event_id to numeric if possible
    if len(events_df) > 0:
        try:
            events_df['event_id'] = pd.to_numeric(events_df['event_id'])
            phases_df['event_id'] = pd.to_numeric(phases_df['event_id'])
        except:
            pass  # Keep as string if conversion fails
    
    return events_df, phases_df

# Parse the phase file with the correct format
print("Parsing phase file with correct format...")
#events_df, phases_df = parse_phase_file_correct('../data/ax.hinv.pha.shots_erup')
events_df, phases_df = parse_phase_file_correct('../data/old_catalog_erup')

print(f"\\nParsing complete!")
print(f"Found {len(events_df)} events")
print(f"Found {len(phases_df)} phase picks")

if len(events_df) > 0:
    print(f"\\nFirst 5 events:")
    print(events_df[['event_id', 'datetime_str', 'lat', 'lon', 'depth', 'magnitude']].head())

if len(phases_df) > 0:
    print(f"\\nFirst 10 phase picks:")
    print(phases_df[['event_id', 'station', 'arrival_time', 'weight', 'phase_type', 'quality']].head(10))

Parsing phase file with correct format...
Processing 800 lines...
\nParsing complete!
Found 67 events
Found 732 phase picks
\nFirst 5 events:
   event_id              datetime_str       lat        lon  depth  magnitude
0     0.033  2015-01-22T00:00:27.537Z  45.94689 -129.99863  0.547       -0.3
1     0.093  2015-01-22T00:08:58.891Z  45.94934 -129.99501  0.000        0.0
2     0.030  2015-01-22T02:02:41.042Z  45.94109 -130.01480  0.675       -0.0
3     0.065  2015-01-22T02:19:56.897Z  45.91550 -129.95985  1.598        0.3
4     0.058  2015-01-22T02:36:07.075Z  45.94662 -129.99509  0.509       -0.1
\nFirst 10 phase picks:
   event_id station  arrival_time  weight phase_type quality
0     0.033   AXCC1         0.424    1.00          P        
1     0.033   AXEC1         0.619    0.50          P        
2     0.033   AXEC2         0.664    0.75          P        
3     0.033   AXCC1         1.209    0.25          S        
4     0.033   AXEC1         0.979    0.25          S        
5     

In [10]:
# Cross-correlate events_df and phases_df to create final catalog
final_catalog = pd.merge(phases_df, events_df, on='event_id', suffixes=('_phase', '_event'))

In [12]:
final_catalog.to_csv('../data/old_catalog.csv', index=False)

In [4]:
# CORRECT APPROACH: Create final dataframe with catalog earthquakes x stations x phase picks
print("Creating final dataframe: Catalog earthquakes × Stations × Phase picks")
print("=" * 75)

def create_catalog_station_phase_dataframe(catalog, events_df, phases_df, time_tolerance_seconds=5):
    """
    Create the final dataframe structure:
    - Start with catalog earthquakes (authoritative)
    - Match to events_df by time
    - Create one row per catalog earthquake per station
    - Include phase pick information (P and S waves) for each station
    
    Returns:
    --------
    final_df : pandas.DataFrame
        Each row = one catalog earthquake at one station with phase pick info
    """
    
    print("Step 1: Match catalog earthquakes to events_df by time...")
    
    # Create datetime columns if needed
    if 'datetime' not in catalog.columns:
        catalog['datetime'] = pd.to_datetime(catalog[['year', 'month', 'day', 'hour', 'minute', 'second']])
    if 'datetime' not in events_df.columns:
        events_df['datetime'] = pd.to_datetime(events_df['datetime_str']).dt.tz_localize(None)
    
    # Find matches between catalog and events_df
    catalog_to_events_matches = []
    
    for cat_idx, cat_event in catalog.iterrows():
        cat_time = cat_event['datetime']
        
        # Find closest event in events_df within tolerance
        time_diffs = abs(events_df['datetime'] - cat_time).dt.total_seconds()
        within_tolerance = time_diffs <= time_tolerance_seconds
        
        if within_tolerance.any():
            closest_idx = time_diffs.idxmin()
            closest_event = events_df.loc[closest_idx]
            
            catalog_to_events_matches.append({
                'catalog_idx': cat_idx,
                'events_df_idx': closest_idx,
                'phase_event_id': closest_event['event_id'],
                'time_diff_seconds': time_diffs.loc[closest_idx]
            })
        
        if (cat_idx + 1) % 25000 == 0:
            print(f"  Processed {cat_idx + 1:,} catalog events...")
    
    matches_df = pd.DataFrame(catalog_to_events_matches)
    print(f"Found {len(matches_df)} catalog events with matching phase data")
    
    print("\nStep 2: Create final dataframe with catalog earthquakes × stations...")
    
    # Get all unique stations
    all_stations = phases_df['station'].unique()
    print(f"Stations in network: {sorted(all_stations)}")
    
    final_rows = []
    
    for _, match in matches_df.iterrows():
        catalog_idx = match['catalog_idx']
        phase_event_id = match['phase_event_id']
        
        # Get catalog earthquake info
        cat_earthquake = catalog.loc[catalog_idx]
        
        # Get all phase picks for this earthquake
        earthquake_phases = phases_df[phases_df['event_id'] == phase_event_id]
        
        # Group phase picks by station
        for station in all_stations:
            station_phases = earthquake_phases[earthquake_phases['station'] == station]
            
            # Initialize row with catalog earthquake info
            row = {
                # Catalog earthquake information (authoritative)
                'catalog_idx': catalog_idx,
                'catalog_id': cat_earthquake.get('id', catalog_idx),
                'catalog_year': cat_earthquake['year'],
                'catalog_month': cat_earthquake['month'],
                'catalog_day': cat_earthquake['day'],
                'catalog_hour': cat_earthquake['hour'],
                'catalog_minute': cat_earthquake['minute'],
                'catalog_second': cat_earthquake['second'],
                'catalog_datetime': cat_earthquake['datetime'],
                'catalog_lat': cat_earthquake['lat'],
                'catalog_lon': cat_earthquake['lon'],
                'catalog_depth': cat_earthquake['dep'],
                'catalog_mag': cat_earthquake['mag'],
                
                # Station information
                'station': station,
                'phase_event_id': phase_event_id,
                'time_diff_seconds': match['time_diff_seconds'],
                
                # Initialize phase pick fields
                'has_data': len(station_phases) > 0,
                'total_picks': len(station_phases),
                'p_arrival_time': None,
                'p_weight': None,
                'p_quality': None,
                's_arrival_time': None,
                's_weight': None,
                's_quality': None
            }
            
            # Fill in phase pick information if available
            if len(station_phases) > 0:
                # P-wave information
                p_phases = station_phases[station_phases['phase_type'] == 'P']
                if len(p_phases) > 0:
                    p_pick = p_phases.iloc[0]  # Take first P pick if multiple
                    row['p_arrival_time'] = p_pick['arrival_time']
                    row['p_weight'] = p_pick['weight']
                    row['p_quality'] = p_pick['quality']
                
                # S-wave information
                s_phases = station_phases[station_phases['phase_type'] == 'S']
                if len(s_phases) > 0:
                    s_pick = s_phases.iloc[0]  # Take first S pick if multiple
                    row['s_arrival_time'] = s_pick['arrival_time']
                    row['s_weight'] = s_pick['weight']
                    row['s_quality'] = s_pick['quality']
            
            final_rows.append(row)
    
    return pd.DataFrame(final_rows)

# Create the final dataframe
print("\nCreating final catalog × stations × phase picks dataframe...")
final_catalog_station_df = create_catalog_station_phase_dataframe(
    catalog, events_df, phases_df, time_tolerance_seconds=5
)

print(f"\nFinal dataframe structure:")
print(f"• Total rows: {len(final_catalog_station_df):,}")
print(f"• Unique catalog earthquakes: {final_catalog_station_df['catalog_idx'].nunique():,}")
print(f"• Unique stations: {final_catalog_station_df['station'].nunique()}")
print(f"• Rows with phase data: {final_catalog_station_df['has_data'].sum():,}")
print(f"• Rows without phase data: {(~final_catalog_station_df['has_data']).sum():,}")

print(f"\nStructure: Each row = one catalog earthquake at one station")
print(f"Expected total rows = catalog earthquakes × stations = {len(catalog)} × {final_catalog_station_df['station'].nunique()} = {len(catalog) * final_catalog_station_df['station'].nunique():,}")

# Show sample data
print(f"\nSample rows (first 10):")
display_cols = ['catalog_idx', 'catalog_datetime', 'catalog_lat', 'catalog_lon', 'catalog_mag',
                'station', 'has_data', 'p_arrival_time', 's_arrival_time', 'p_weight', 's_weight']
print(final_catalog_station_df[display_cols].head(10))

# Show statistics by station
print(f"\nData availability by station:")
station_stats = final_catalog_station_df.groupby('station').agg({
    'has_data': ['count', 'sum'],
    'p_arrival_time': lambda x: x.notna().sum(),
    's_arrival_time': lambda x: x.notna().sum()
}).round(1)

station_stats.columns = ['total_earthquakes', 'with_data', 'p_picks', 's_picks']
station_stats['data_rate_%'] = (station_stats['with_data'] / station_stats['total_earthquakes'] * 100).round(1)
print(station_stats)

# Example: Show all stations for one earthquake
first_eq_idx = final_catalog_station_df['catalog_idx'].iloc[0]
first_eq_data = final_catalog_station_df[final_catalog_station_df['catalog_idx'] == first_eq_idx]
print(f"\nExample - All stations for catalog earthquake {first_eq_idx}:")
example_cols = ['station', 'has_data', 'p_arrival_time', 's_arrival_time', 'p_weight', 's_weight', 'p_quality', 's_quality']
print(first_eq_data[example_cols])

print(f"\nThis is the correct structure for shear-wave splitting analysis!")
print(f"• Each catalog earthquake is represented at each station")
print(f"• Phase pick information (P and S waves) is available when data exists") 
print(f"• Missing data is clearly marked (has_data = False)")
print(f"• Ready for geometric calculations and splitting analysis")

Creating final dataframe: Catalog earthquakes × Stations × Phase picks

Creating final catalog × stations × phase picks dataframe...
Step 1: Match catalog earthquakes to events_df by time...
  Processed 25,000 catalog events...
  Processed 50,000 catalog events...
  Processed 75,000 catalog events...
  Processed 100,000 catalog events...
  Processed 125,000 catalog events...
Found 44 catalog events with matching phase data

Step 2: Create final dataframe with catalog earthquakes × stations...
Stations in network: ['AXAS1', 'AXAS2', 'AXCC1', 'AXEC1', 'AXEC2', 'AXEC3', 'AXID1']

Final dataframe structure:
• Total rows: 308
• Unique catalog earthquakes: 44
• Unique stations: 7
• Rows with phase data: 281
• Rows without phase data: 27

Structure: Each row = one catalog earthquake at one station
Expected total rows = catalog earthquakes × stations = 144328 × 7 = 1,010,296

Sample rows (first 10):
   catalog_idx        catalog_datetime  catalog_lat  catalog_lon  catalog_mag  \
0      10348.0

In [5]:
final_catalog_station_df.head()

,catalog_idx,catalog_id,catalog_year,catalog_month,catalog_day,catalog_hour,catalog_minute,catalog_second,catalog_datetime,catalog_lat,...,phase_event_id,time_diff_seconds,has_data,total_picks,p_arrival_time,p_weight,p_quality,s_arrival_time,s_weight,s_quality
0,10348.0,1024515,2015,1,22,2,2,41.038,2015-01-22 02:02:41.038,45.94132,...,0.03,0.004,True,2,0.428,0.37,,1.403,0.17,
1,10348.0,1024515,2015,1,22,2,2,41.038,2015-01-22 02:02:41.038,45.94132,...,0.03,0.004,True,2,0.668,0.25,,1.493,0.25,
2,10348.0,1024515,2015,1,22,2,2,41.038,2015-01-22 02:02:41.038,45.94132,...,0.03,0.004,True,2,0.853,0.50,,1.628,0.33,
3,10348.0,1024515,2015,1,22,2,2,41.038,2015-01-22 02:02:41.038,45.94132,...,0.03,0.004,True,2,0.828,1.00,,1.608,0.33,
4,10348.0,1024515,2015,1,22,2,2,41.038,2015-01-22 02:02:41.038,45.94132,...,0.03,0.004,True,2,0.583,0.25,,1.178,0.12,


In [7]:
final_catalog_station_df.to_csv('../data/old_catalog.csv', index=False)